# CrisLens — Fact-Check Retrieval System
## SemEval 2025 Task 7: Multilingual Fact-Check Post Retrieval

**Pipeline:**
1. Load & parse dataset
2. Build training pairs (crosslingual + monolingual)
3. Fine-tune `paraphrase-multilingual-mpnet-base-v2` for both tasks
4. Encode fact-checks & build FAISS indexes
5. Evaluate with Success@10 metric
6. Interactive demo & HuggingFace upload

## 1. Setup & Dependencies

In [ ]:
!pip install sentence-transformers faiss-cpu -q

import os
import json
import ast
import shutil
import gc
import torch
import numpy as np
import pandas as pd
import faiss
from tqdm import tqdm
from sentence_transformers import SentenceTransformer, InputExample
from sentence_transformers.losses import MultipleNegativesRankingLoss
from torch.utils.data import DataLoader

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

OUTPUT = '/kaggle/working/'

# ── Verify GPU ──
print("="*40)
print(f"GPU count : {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f"GPU {i}    : {torch.cuda.get_device_name(i)}")
    print(f"  VRAM  : {props.total_memory / 1e9:.1f} GB")
print("="*40)
print("Setup complete ✓")

## 2. Load Dataset

In [ ]:
# ── Paths ──
DATA     = '/kaggle/input/datasets/sayyamsatti/samevel-2025-task-7/'
TRAIN    = DATA + 'train_dev_sets/'
TEST     = DATA + 'test_set/'
SCRIPTS  = DATA + 'scripts/'

# ── Load train/dev files ──
print("Loading train/dev files...")
train_posts    = pd.read_csv(TRAIN + 'posts.csv')
train_fc       = pd.read_csv(TRAIN + 'fact_checks.csv')
train_pairs    = pd.read_csv(TRAIN + 'pairs.csv')
dev_pairs_cross = pd.read_csv(TRAIN + 'pairs_dev_crosslingual.csv')
dev_pairs_mono  = pd.read_csv(TRAIN + 'pairs_dev_monolingual.csv')

with open(TRAIN + 'tasks.json', 'r') as f:
    train_tasks = json.load(f)
with open(TRAIN + 'crosslingual_reference.json', 'r') as f:
    cross_ref = json.load(f)
with open(TRAIN + 'monolingual_reference.json', 'r') as f:
    mono_ref = json.load(f)

print("Train/dev loaded ✓")

# ── Load test files ──
print("\nLoading test files...")
test_posts = pd.read_csv(TEST + 'posts.csv')
test_fc    = pd.read_csv(TEST + 'fact_checks.csv')
test_pairs_cross = pd.read_csv(TEST + 'pairs_test_crosslingual.csv')
test_pairs_mono  = pd.read_csv(TEST + 'pairs_test_monolingual.csv')

with open(TEST + 'tasks.json', 'r') as f:
    test_tasks = json.load(f)
with open(TEST + 'crosslingual_reference.json', 'r') as f:
    test_cross_ref = json.load(f)
with open(TEST + 'monolingual_reference.json', 'r') as f:
    test_mono_ref = json.load(f)

print("Test files loaded ✓")

# ── Summary ──
print("\n" + "="*45)
print(f"train_posts      : {len(train_posts):>8,} rows")
print(f"train_fc         : {len(train_fc):>8,} rows")
print(f"train_pairs      : {len(train_pairs):>8,} rows")
print(f"dev_pairs_cross  : {len(dev_pairs_cross):>8,} rows")
print(f"dev_pairs_mono   : {len(dev_pairs_mono):>8,} rows")
print(f"test_posts       : {len(test_posts):>8,} rows")
print(f"test_fc          : {len(test_fc):>8,} rows")
print(f"test_pairs_cross : {len(test_pairs_cross):>8,} rows")
print(f"test_pairs_mono  : {len(test_pairs_mono):>8,} rows")
print("="*45)
print("\nAll files loaded ✓")

## 3. Parse Text Fields

In [ ]:
def parse_text_tuple(val):
    """
    Parses the tuple format in posts/fact_checks:
    ('original text', 'english translation', [('lang', confidence)])
    Returns: (original, english, language_code)
    """
    if pd.isna(val):
        return None, None, 'unk'
    try:
        t = ast.literal_eval(str(val))
        original = t[0] if t[0] else None
        english  = t[1] if t[1] else None
        lang     = t[2][0][0] if t[2] else 'unk'
        return original, english, lang
    except:
        return str(val), str(val), 'unk'

def parse_ocr(val):
    """
    Parses OCR field from posts — list of tuples.
    Returns combined OCR text in english.
    """
    if pd.isna(val): return ''
    try:
        items = ast.literal_eval(str(val))
        texts = []
        for item in items:
            if item[1]: texts.append(item[1])  # english version
            elif item[0]: texts.append(item[0]) # original if no english
        return ' '.join(texts).strip()
    except:
        return ''

# ── Parse train posts ──
print("Parsing train posts...")
train_posts[['text_orig','text_eng','lang']] = train_posts['text'].apply(
    lambda x: pd.Series(parse_text_tuple(x)))
train_posts['ocr_eng'] = train_posts['ocr'].apply(parse_ocr)
train_posts['post_text_orig'] = (
    train_posts['text_orig'].fillna('') + ' ' +
    train_posts['ocr_eng'].fillna('')
).str.strip()
train_posts['post_text_eng'] = (
    train_posts['text_eng'].fillna('') + ' ' +
    train_posts['ocr_eng'].fillna('')
).str.strip()
print(f"  Train posts parsed ✓ — {len(train_posts):,} rows")

# ── Parse test posts ──
print("Parsing test posts...")
test_posts[['text_orig','text_eng','lang']] = test_posts['text'].apply(
    lambda x: pd.Series(parse_text_tuple(x)))
test_posts['ocr_eng'] = test_posts['ocr'].apply(parse_ocr)
test_posts['post_text_orig'] = (
    test_posts['text_orig'].fillna('') + ' ' +
    test_posts['ocr_eng'].fillna('')
).str.strip()
test_posts['post_text_eng'] = (
    test_posts['text_eng'].fillna('') + ' ' +
    test_posts['ocr_eng'].fillna('')
).str.strip()
print(f"  Test posts parsed ✓ — {len(test_posts):,} rows")

# ── Parse train fact_checks ──
print("Parsing train fact_checks...")
train_fc[['claim_orig','claim_eng','lang']] = train_fc['claim'].apply(
    lambda x: pd.Series(parse_text_tuple(x)))
train_fc[['title_orig','title_eng','_']] = train_fc['title'].apply(
    lambda x: pd.Series(parse_text_tuple(x)))
train_fc['fc_text_orig'] = (
    train_fc['claim_orig'].fillna('') + ' ' +
    train_fc['title_orig'].fillna('')
).str.strip()
train_fc['fc_text_eng'] = (
    train_fc['claim_eng'].fillna('') + ' ' +
    train_fc['title_eng'].fillna('')
).str.strip()
print(f"  Train fact_checks parsed ✓ — {len(train_fc):,} rows")

# ── Parse test fact_checks ──
print("Parsing test fact_checks...")
test_fc[['claim_orig','claim_eng','lang']] = test_fc['claim'].apply(
    lambda x: pd.Series(parse_text_tuple(x)))
test_fc[['title_orig','title_eng','_']] = test_fc['title'].apply(
    lambda x: pd.Series(parse_text_tuple(x)))
test_fc['fc_text_orig'] = (
    test_fc['claim_orig'].fillna('') + ' ' +
    test_fc['title_orig'].fillna('')
).str.strip()
test_fc['fc_text_eng'] = (
    test_fc['claim_eng'].fillna('') + ' ' +
    test_fc['title_eng'].fillna('')
).str.strip()
print(f"  Test fact_checks parsed ✓ — {len(test_fc):,} rows")
print("\nAll parsing complete ✓")

## 4. Save Parsed Files & Reference Data

In [ ]:
print("Saving parsed files to /kaggle/working/...")

# ── Save parsed posts ──
train_posts.to_csv(OUTPUT + 'train_posts_parsed.csv', index=False)
print(f"  train_posts_parsed.csv    ✓ — {len(train_posts):,} rows")

test_posts.to_csv(OUTPUT + 'test_posts_parsed.csv', index=False)
print(f"  test_posts_parsed.csv     ✓ — {len(test_posts):,} rows")

# ── Save parsed fact_checks ──
train_fc.to_csv(OUTPUT + 'train_fc_parsed.csv', index=False)
print(f"  train_fc_parsed.csv       ✓ — {len(train_fc):,} rows")

test_fc.to_csv(OUTPUT + 'test_fc_parsed.csv', index=False)
print(f"  test_fc_parsed.csv        ✓ — {len(test_fc):,} rows")

# ── Save pairs ──
train_pairs.to_csv(OUTPUT + 'train_pairs.csv', index=False)
dev_pairs_cross.to_csv(OUTPUT + 'dev_pairs_cross.csv', index=False)
dev_pairs_mono.to_csv(OUTPUT + 'dev_pairs_mono.csv', index=False)
test_pairs_cross.to_csv(OUTPUT + 'test_pairs_cross.csv', index=False)
test_pairs_mono.to_csv(OUTPUT + 'test_pairs_mono.csv', index=False)
print(f"  pair files                ✓ — 5 files")

# ── Save reference jsons ──
shutil.copy(TRAIN + 'crosslingual_reference.json', OUTPUT + 'dev_cross_reference.json')
shutil.copy(TRAIN + 'monolingual_reference.json',  OUTPUT + 'dev_mono_reference.json')
shutil.copy(TEST  + 'crosslingual_reference.json', OUTPUT + 'test_cross_reference.json')
shutil.copy(TEST  + 'monolingual_reference.json',  OUTPUT + 'test_mono_reference.json')
print(f"  reference json files      ✓ — 4 files")

# ── Copy scoring scripts ──
shutil.copy(SCRIPTS + 'crosslingual_scoring.py', OUTPUT + 'crosslingual_scoring.py')
shutil.copy(SCRIPTS + 'monolingual_scoring.py',  OUTPUT + 'monolingual_scoring.py')
print(f"  scoring scripts           ✓ — 2 files")

print("\nAll saved ✓")

## 5. Build Training Data

In [ ]:
# ── Build complete pairs dataframe ──
pairs_with_lang = train_pairs.merge(
    train_posts[['post_id','lang','post_text_orig','post_text_eng']],
    on='post_id', how='left'
).merge(
    train_fc[['fact_check_id','lang','fc_text_orig','fc_text_eng']],
    on='fact_check_id',
    suffixes=('_post','_fc'),
    how='left'
)

# ── Drop rows with missing text ──
pairs_with_lang = pairs_with_lang.dropna(
    subset=['post_text_eng','fc_text_eng']
).reset_index(drop=True)

print(f"Total pairs after dropping nulls: {len(pairs_with_lang):,}")

# ────────────────────────────────────────
# CROSSLINGUAL TRAINING DATA
# Use ONLY english translations
# ────────────────────────────────────────
cross_train = pairs_with_lang[['post_id','fact_check_id',
                                'post_text_eng','fc_text_eng',
                                'lang_post','lang_fc']].copy()
cross_train.columns = ['post_id','fact_check_id',
                       'query','positive',
                       'lang_post','lang_fc']

cross_train = cross_train[
    (cross_train['query'].str.strip().str.len() > 5) &
    (cross_train['positive'].str.strip().str.len() > 5)
].reset_index(drop=True)

print(f"\nCrosslingual training pairs : {len(cross_train):,}")

# ────────────────────────────────────────
# MONOLINGUAL TRAINING DATA
# Use original + english translation
# ────────────────────────────────────────
mono_orig = pairs_with_lang[['post_id','fact_check_id',
                              'post_text_orig','fc_text_orig',
                              'lang_post','lang_fc']].copy()
mono_orig.columns = ['post_id','fact_check_id',
                     'query','positive',
                     'lang_post','lang_fc']

mono_eng = pairs_with_lang[['post_id','fact_check_id',
                             'post_text_eng','fc_text_eng',
                             'lang_post','lang_fc']].copy()
mono_eng.columns = ['post_id','fact_check_id',
                    'query','positive',
                    'lang_post','lang_fc']

mono_train = pd.concat([mono_orig, mono_eng], ignore_index=True)
mono_train = mono_train[
    (mono_train['query'].str.strip().str.len() > 5) &
    (mono_train['positive'].str.strip().str.len() > 5)
].reset_index(drop=True)

print(f"Monolingual training pairs  : {len(mono_train):,}")
print("(original + english = 2x data)")

# ── Save both training sets ──
cross_train.to_csv(OUTPUT + 'cross_train_data.csv', index=False)
mono_train.to_csv(OUTPUT + 'mono_train_data.csv', index=False)

print("\n" + "="*45)
print(f"cross_train_data.csv saved : {len(cross_train):,} pairs")
print(f"mono_train_data.csv saved  : {len(mono_train):,} pairs")
print("="*45)
print("Training data ready ✓")

## 6. Fine-tune Crosslingual Model

In [ ]:
MODEL_NAME = 'sentence-transformers/paraphrase-multilingual-mpnet-base-v2'

gc.collect()
torch.cuda.empty_cache()

# ── Load training data ──
cross_train = pd.read_csv(OUTPUT + 'cross_train_data.csv')
print(f"Loading model: {MODEL_NAME}")

model = SentenceTransformer(
    MODEL_NAME,
    cache_folder=OUTPUT + 'model_cache/',
    device='cuda:0'
)
print(f"Model loaded ✓ — Dim: {model.get_sentence_embedding_dimension()}")

# ── Build examples ──
train_examples = []
for _, row in cross_train.iterrows():
    q = str(row['query']).strip()
    p = str(row['positive']).strip()
    if len(q) > 5 and len(p) > 5:
        train_examples.append(InputExample(texts=[q, p]))
print(f"Examples: {len(train_examples):,}")

# ── DataLoader & Loss ──
train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=32)
train_loss = MultipleNegativesRankingLoss(model)

print(f"Steps: {len(train_dataloader):,}")
print("Starting crosslingual training...")

# ── Train ──
model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=1,
    warmup_steps=100,
    use_amp=False,
    optimizer_params={'lr': 2e-5},
    show_progress_bar=True
)

# ── Verify & Save ──
print("\nVerifying model...")
test_emb = model.encode(
    ["flood evacuation emergency"],
    normalize_embeddings=True,
    convert_to_numpy=True
)

if np.isnan(test_emb).any():
    print("ERROR: NaN detected — model NOT saved")
else:
    model.save(OUTPUT + 'cross_model_mpnet/')
    print("Crosslingual model saved ✓ — no NaN")

## 7. Fine-tune Monolingual Model

In [ ]:
del model
gc.collect()
torch.cuda.empty_cache()

# ── Load monolingual training data ──
mono_train = pd.read_csv(OUTPUT + 'mono_train_data.csv')

train_examples_mono = []
for _, row in mono_train.iterrows():
    q = str(row['query']).strip()
    p = str(row['positive']).strip()
    if len(q) > 5 and len(p) > 5:
        train_examples_mono.append(InputExample(texts=[q, p]))
print(f"Examples: {len(train_examples_mono):,}")

# ── Load fresh model ──
model_mono = SentenceTransformer(
    MODEL_NAME,
    cache_folder=OUTPUT + 'model_cache/',
    device='cuda:0'
)
print("Model loaded ✓")

# ── DataLoader & Loss ──
train_dataloader_mono = DataLoader(train_examples_mono, shuffle=True, batch_size=32)
train_loss_mono = MultipleNegativesRankingLoss(model_mono)

print(f"Steps: {len(train_dataloader_mono):,}")
print("Starting monolingual training...")

# ── Train ──
model_mono.fit(
    train_objectives=[(train_dataloader_mono, train_loss_mono)],
    epochs=1,
    warmup_steps=100,
    use_amp=False,
    optimizer_params={'lr': 2e-5},
    show_progress_bar=True
)

# ── Verify & Save ──
print("\nVerifying model...")
test_emb = model_mono.encode(
    ["flood evacuation emergency"],
    normalize_embeddings=True,
    convert_to_numpy=True
)

if np.isnan(test_emb).any():
    print("ERROR: NaN detected — model NOT saved")
else:
    model_mono.save(OUTPUT + 'mono_model_mpnet/')
    print("Monolingual model saved ✓ — no NaN")

del model_mono
gc.collect()
torch.cuda.empty_cache()

## 8. Encode Fact-Checks & Build FAISS Indexes

In [ ]:
gc.collect()
torch.cuda.empty_cache()

# ── Load fact-checks ──
train_fc = pd.read_csv(OUTPUT + 'train_fc_parsed.csv')
test_fc  = pd.read_csv(OUTPUT + 'test_fc_parsed.csv')
all_fc   = pd.concat([train_fc, test_fc], ignore_index=True)
all_fc   = all_fc.drop_duplicates(subset=['fact_check_id']).reset_index(drop=True)
all_fc['fc_text_eng']  = all_fc['fc_text_eng'].fillna('unknown')
all_fc['fc_text_orig'] = all_fc['fc_text_orig'].fillna('unknown')
print(f"Fact-checks: {len(all_fc):,}")

# ── Encode with crosslingual model ──
print("\nLoading crosslingual model...")
model_cross = SentenceTransformer(OUTPUT + 'cross_model_mpnet/', device='cuda:0')

print("Encoding crosslingual (english)...")
cross_embs = model_cross.encode(
    all_fc['fc_text_eng'].tolist(),
    batch_size=512,
    normalize_embeddings=True,
    show_progress_bar=True,
    convert_to_numpy=True
).astype('float32')

print(f"NaN check: {np.isnan(cross_embs).any()}")
np.save(OUTPUT + 'cross_fc_embeddings_mpnet.npy', cross_embs)
print(f"Saved ✓ shape: {cross_embs.shape}")

del model_cross, cross_embs
gc.collect()
torch.cuda.empty_cache()

# ── Encode with monolingual model ──
print("\nLoading monolingual model...")
model_mono = SentenceTransformer(OUTPUT + 'mono_model_mpnet/', device='cuda:0')

print("Encoding monolingual (original)...")
mono_embs = model_mono.encode(
    all_fc['fc_text_orig'].tolist(),
    batch_size=512,
    normalize_embeddings=True,
    show_progress_bar=True,
    convert_to_numpy=True
).astype('float32')

print(f"NaN check: {np.isnan(mono_embs).any()}")
np.save(OUTPUT + 'mono_fc_embeddings_mpnet.npy', mono_embs)
print(f"Saved ✓ shape: {mono_embs.shape}")

fc_ids = all_fc['fact_check_id'].tolist()
np.save(OUTPUT + 'fc_ids.npy', np.array(fc_ids))

del model_mono, mono_embs
gc.collect()
torch.cuda.empty_cache()

# ── Build FAISS indexes ──
print("\nBuilding FAISS indexes...")
cross_embs = np.load(OUTPUT + 'cross_fc_embeddings_mpnet.npy').astype('float32')
cross_index = faiss.IndexFlatIP(cross_embs.shape[1])
cross_index.add(cross_embs)
faiss.write_index(cross_index, OUTPUT + 'cross_faiss_mpnet.index')
print(f"Cross FAISS: {cross_index.ntotal:,} vectors ✓")
del cross_embs, cross_index

mono_embs = np.load(OUTPUT + 'mono_fc_embeddings_mpnet.npy').astype('float32')
mono_index = faiss.IndexFlatIP(mono_embs.shape[1])
mono_index.add(mono_embs)
faiss.write_index(mono_index, OUTPUT + 'mono_faiss_mpnet.index')
print(f"Mono FAISS : {mono_index.ntotal:,} vectors ✓")
del mono_embs, mono_index

print("\n" + "="*40)
print("All encodings & FAISS indexes complete ✓")
print("="*40)

## 9. Evaluation (Dev + Test)

In [ ]:
# ── Load models and indexes ──
print("Loading...")
model_cross = SentenceTransformer(OUTPUT + 'cross_model_mpnet/', device='cuda:0')
model_mono  = SentenceTransformer(OUTPUT + 'mono_model_mpnet/',  device='cuda:0')

fc_ids      = np.load(OUTPUT + 'fc_ids.npy', allow_pickle=True)
fc_ids_list = [int(x) for x in fc_ids]

cross_index = faiss.read_index(OUTPUT + 'cross_faiss_mpnet.index')
mono_index  = faiss.read_index(OUTPUT + 'mono_faiss_mpnet.index')

train_posts = pd.read_csv(OUTPUT + 'train_posts_parsed.csv')
test_posts  = pd.read_csv(OUTPUT + 'test_posts_parsed.csv')
all_posts   = pd.concat([train_posts, test_posts], ignore_index=True)

dev_pairs_cross  = pd.read_csv(OUTPUT + 'dev_pairs_cross.csv')
dev_pairs_mono   = pd.read_csv(OUTPUT + 'dev_pairs_mono.csv')
test_pairs_cross = pd.read_csv(OUTPUT + 'test_pairs_cross.csv')
test_pairs_mono  = pd.read_csv(OUTPUT + 'test_pairs_mono.csv')

with open(OUTPUT + 'dev_cross_reference.json')  as f: dev_cross_ref  = json.load(f)
with open(OUTPUT + 'dev_mono_reference.json')   as f: dev_mono_ref   = json.load(f)
with open(OUTPUT + 'test_cross_reference.json') as f: test_cross_ref = json.load(f)
with open(OUTPUT + 'test_mono_reference.json')  as f: test_mono_ref  = json.load(f)

print("All loaded ✓")

# ── Retrieval functions ──
def retrieve_cross(query, top_k=10):
    q_vec = model_cross.encode(
        [query], normalize_embeddings=True,
        convert_to_numpy=True).astype('float32')
    _, I = cross_index.search(q_vec, top_k)
    return [fc_ids_list[i] for i in I[0]]

def retrieve_mono(query, top_k=10):
    q_vec = model_mono.encode(
        [query], normalize_embeddings=True,
        convert_to_numpy=True).astype('float32')
    _, I = mono_index.search(q_vec, top_k)
    return [fc_ids_list[i] for i in I[0]]

def success_at_k(results, reference, k=10):
    scores = []
    for post_id, retrieved in results.items():
        if post_id not in reference: continue
        relevant = set(reference[post_id])
        hit = 1 if len(relevant & set(retrieved[:k])) > 0 else 0
        scores.append(hit)
    return round(sum(scores)/len(scores), 4) if scores else 0

# ── DEV evaluation ──
print("\nRunning DEV crosslingual...")
dev_cross_results = {}
for pid in tqdm(dev_pairs_cross['post_id'].unique()):
    row = all_posts[all_posts['post_id']==pid]
    if row.empty: continue
    q = str(row.iloc[0]['post_text_eng']).strip()
    dev_cross_results[str(pid)] = retrieve_cross(q)

print("Running DEV monolingual...")
dev_mono_results = {}
for pid in tqdm(dev_pairs_mono['post_id'].unique()):
    row = all_posts[all_posts['post_id']==pid]
    if row.empty: continue
    q = str(row.iloc[0]['post_text_orig']).strip()
    dev_mono_results[str(pid)] = retrieve_mono(q)

# ── TEST evaluation ──
print("\nRunning TEST crosslingual...")
test_cross_results = {}
for pid in tqdm(test_pairs_cross['post_id'].unique()):
    row = test_posts[test_posts['post_id']==pid]
    if row.empty: continue
    q = str(row.iloc[0]['post_text_eng']).strip()
    test_cross_results[str(pid)] = retrieve_cross(q)

print("Running TEST monolingual...")
test_mono_results = {}
for pid in tqdm(test_pairs_mono['post_id'].unique()):
    row = test_posts[test_posts['post_id']==pid]
    if row.empty: continue
    q = str(row.iloc[0]['post_text_orig']).strip()
    test_mono_results[str(pid)] = retrieve_mono(q)

# ── Results ──
dev_cross_s10  = success_at_k(dev_cross_results,  dev_cross_ref)
dev_mono_s10   = success_at_k(dev_mono_results,   dev_mono_ref)
test_cross_s10 = success_at_k(test_cross_results, test_cross_ref)
test_mono_s10  = success_at_k(test_mono_results,  test_mono_ref)

print("\n" + "="*50)
print("FINAL EVALUATION RESULTS")
print("="*50)
print("                   DEV      TEST")
print(f"Crosslingual S@10: {dev_cross_s10}    {test_cross_s10}")
print(f"Monolingual  S@10: {dev_mono_s10}    {test_mono_s10}")
print("="*50)

## 10. Interactive Demo

In [ ]:
# ── Build fact-check lookup ──
train_fc_df = pd.read_csv(OUTPUT + 'train_fc_parsed.csv')
test_fc_df  = pd.read_csv(OUTPUT + 'test_fc_parsed.csv')
all_fc_df   = pd.concat([train_fc_df, test_fc_df], ignore_index=True)
all_fc_df   = all_fc_df.drop_duplicates(subset=['fact_check_id']).reset_index(drop=True)
fc_lookup   = dict(zip(all_fc_df['fact_check_id'], all_fc_df['fc_text_eng']))

def search(query, mode='cross', top_k=5):
    """Search for fact-checks matching a query."""
    if mode == 'cross':
        q_vec = model_cross.encode(
            [query], normalize_embeddings=True,
            convert_to_numpy=True).astype('float32')
        _, I = cross_index.search(q_vec, top_k)
    else:
        q_vec = model_mono.encode(
            [query], normalize_embeddings=True,
            convert_to_numpy=True).astype('float32')
        _, I = mono_index.search(q_vec, top_k)

    print("\n" + "="*60)
    print("MODE  : " + mode.upper())
    print("QUERY : " + query)
    print("="*60)
    for rank, idx in enumerate(I[0], 1):
        fc_id = fc_ids_list[idx]
        text  = fc_lookup.get(fc_id, 'N/A')
        print(f"\n#{rank} [ID: {fc_id}]")
        print(f"   {text}")

# ── Test queries ──
search("5G towers spreading coronavirus", mode='cross')
search("COVID vaccine causes infertility", mode='cross')
search("floods in Germany killed babies", mode='cross')
search("La vacuna COVID causa infertilidad", mode='mono')
search("Les tours 5G propagent le coronavirus", mode='mono')

## 11. Upload to HuggingFace

In [ ]:
import subprocess
subprocess.run(['pip', 'install', 'huggingface_hub', 'sentence-transformers', '-q'])

from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi, login

# ── Auth ──
secrets = UserSecretsClient()
HF_TOKEN = secrets.get_secret("HF_TOKEN")
login(token=HF_TOKEN)
api = HfApi(token=HF_TOKEN)
print("HuggingFace logged in ✓")

HF_USERNAME = "Sayyam-1"

# ── Push crosslingual model ──
print("\nUploading crosslingual model...")
model_cross_upload = SentenceTransformer(OUTPUT + 'cross_model_mpnet/')
model_cross_upload.push_to_hub(
    f"{HF_USERNAME}/crislens-cross-mpnet",
    token=HF_TOKEN
)
print("cross_model_mpnet ✓")
del model_cross_upload

# ── Push monolingual model ──
print("\nUploading monolingual model...")
model_mono_upload = SentenceTransformer(OUTPUT + 'mono_model_mpnet/')
model_mono_upload.push_to_hub(
    f"{HF_USERNAME}/crislens-mono-mpnet",
    token=HF_TOKEN
)
print("mono_model_mpnet ✓")
del model_mono_upload

# ── Create dataset repo ──
DATASET_REPO = f"{HF_USERNAME}/crislens-artifacts"
api.create_repo(
    repo_id=DATASET_REPO,
    repo_type="dataset",
    exist_ok=True,
    private=False,
    token=HF_TOKEN
)
print("\nDataset repo created ✓")

# ── Upload artifact files ──
artifacts = [
    "cross_fc_embeddings_mpnet.npy",
    "mono_fc_embeddings_mpnet.npy",
    "cross_faiss_mpnet.index",
    "mono_faiss_mpnet.index",
    "fc_ids.npy",
    "train_posts_parsed.csv",
    "train_fc_parsed.csv",
    "test_posts_parsed.csv",
    "test_fc_parsed.csv",
    "train_pairs.csv",
    "dev_pairs_cross.csv",
    "dev_pairs_mono.csv",
    "test_pairs_cross.csv",
    "test_pairs_mono.csv",
    "dev_cross_reference.json",
    "dev_mono_reference.json",
    "test_cross_reference.json",
    "test_mono_reference.json",
]

print(f"\nUploading {len(artifacts)} files...")

for filename in artifacts:
    path = OUTPUT + filename
    if not os.path.exists(path):
        print(f"  SKIP: {filename}")
        continue
    size_mb = os.path.getsize(path) / 1e6
    print(f"  {filename} ({size_mb:.1f} MB)...", end=" ")
    api.upload_file(
        path_or_fileobj=path,
        path_in_repo=filename,
        repo_id=DATASET_REPO,
        repo_type="dataset",
        token=HF_TOKEN
    )
    print("✓")

print("\n" + "="*50)
print("ALL UPLOADED ✓")
print(f"Models   : huggingface.co/{HF_USERNAME}/crislens-cross-mpnet")
print(f"Artifacts: huggingface.co/datasets/{HF_USERNAME}/crislens-artifacts")
print("="*50)